# pipe_final — un solo notebook, resumible

Preprocesamiento + FE + Estandarizar + Optuna + Entrenamiento final + Submit,
todo en este notebook, con checkpoints en cada etapa. Si el kernel se reinicia
o se corta la VM a mitad de camino, correr el notebook de nuevo desde arriba:
cada sección revisa si su archivo ya existe en el bucket antes de recalcular
("-> RESUME (se saltea)"), así que no se pierde el trabajo ya hecho.

Armado desde cero (no reusa código de `pipe_catedra`/`otro_pipe`/`pipe_nuevo`),
aplicando el mismo patrón de checkpoints que ya se probó en otros intentos del
repo: slug legible por combinación -> archivo marcador -> si existe, se
saltea. Escrituras siempre atómicas (`archivo.tmp` -> rename) para que un kill
a mitad de escritura nunca deje un checkpoint que parezca completo sin estarlo.

Orden: 0) Setup -> 1) Parametros -> 2) Slugs -> 3) Preprocesamiento -> 4) FE
-> 5) Estandarizar -> 6) Optuna -> 7) Entrenamiento final + Submit ->
8) Grilla (recorre todas las combinaciones) -> 9) Leaderboard.


## 0) Setup

In [ ]:
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
import polars as pl
from sklearn.ensemble import RandomForestRegressor


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET    = resolver_bucket()
DIR_RAW   = BUCKET / "datasets"
DIR_OUT   = BUCKET / "pipe_final"
DIR_LOCAL = Path.home() / "pipe_final_local"   # sqlite vivo de Optuna, fuera del FUSE
DIR_RAW.mkdir(parents=True, exist_ok=True)
DIR_OUT.mkdir(parents=True, exist_ok=True)
DIR_LOCAL.mkdir(parents=True, exist_ok=True)

print(f"BUCKET : {BUCKET}")
print(f"crudos : {DIR_RAW}")
print(f"salida : {DIR_OUT}")
print(f"local  : {DIR_LOCAL}")


def escribir_atomico(escribir_fn, path: Path):
    """escribir_fn(tmp_path) escribe en un .tmp y recien al final se renombra
    -> un kill a mitad de escritura nunca deja un checkpoint que parezca
    completo sin estarlo."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    escribir_fn(tmp)
    tmp.replace(path)


def leer_json_reintentando(path: Path, intentos=5, espera=1.0):
    """El bucket via GCS-FUSE a veces tira errores transitorios de I/O al leer."""
    for i in range(intentos):
        try:
            return json.loads(path.read_text())
        except (OSError, json.JSONDecodeError):
            if i == intentos - 1:
                raise
            time.sleep(espera)


In [ ]:
def descargar(archivo):
    url = f"https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{archivo}"
    dst = DIR_RAW / archivo
    if not dst.exists():
        subprocess.run(["wget", url, "-O", str(dst)], check=True)
    print(f"ok: {archivo}")


for _a in ("sell-in.txt.gz", "tb_productos.txt", "product_id_apredecir201912.txt"):
    if (DIR_RAW / _a).exists():
        print(f"ya existe: {_a}")
    else:
        descargar(_a)

# Kaggle auth: usar el de ~/.kaggle si ya existe; si no, buscarlo en el bucket
kaggle_dst = Path.home() / ".kaggle" / "kaggle.json"
kaggle_dst.parent.mkdir(parents=True, exist_ok=True)
if kaggle_dst.exists():
    kaggle_dst.chmod(0o600)
    print("Kaggle auth OK (ya estaba en ~/.kaggle)")
else:
    for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
        if cand.exists():
            shutil.copy(cand, kaggle_dst)
            kaggle_dst.chmod(0o600)
            print(f"Kaggle auth OK (copiado de {cand})")
            break
    else:
        print("aviso: kaggle.json no encontrado. Necesario solo si PARAM[\'submit\']=True.")


## 1) Parametros — un solo dict, toda la combinacion

In [ ]:
PARAM = {
    "modo_agrupacion": "cliente_producto",    # "producto" | "cliente_producto"
    "solo_productos_target": False,            # default: TODOS los productos (pedido explicito)
    "horizonte": 2,

    # FE
    "lags": list(range(0, 12)),
    "ventanas_media_movil": (3, 6),
    "n_vecinos": 3,
    "meses_atras_corte_vecinos": 24,           # corte conservador, sin leakage (vecinos + RF hojas)
    "n_arboles_rf": 50, "profundidad_rf": 6, "min_hoja_rf": 50,
    "niveles_share": ("cat1", "cat2", "cat3"),
    "lags_delta_share": (1, 3),
    "cols_categoricas": ["cat1", "cat2", "cat3", "brand"],

    # Estandarizar
    "ventana_escalado": 6,                     # usada por zscore y rolling_mean

    # grilla a recorrer (metodo_escalado x tipo_target)
    "grilla_metodos_escalado": ("mean", "zscore", "rolling_mean"),
    "grilla_tipos_target": ("nivel", "delta"),

    # Optuna
    "n_trials": 50, "esquema_val": "febreros", "metrica": "wape",
    "objective_lgbm": "regression", "regularizacion": "normal",
    "decay_recencia": None, "backup_cada_n_trials": 10,

    # Entrega
    "semillas_ensemble": [102191], "clip_min": 0.0,
    "kaggle_competition": "labo-iii-2026-rosario", "submit": False,
    "semilla": 102191,
}

print("Parametros:", json.dumps(PARAM, indent=2, default=str))


## 2) Slugs

Nombres legibles (no hashes) para que `ls` en el bucket siga siendo
entendible. Preprocesamiento y FE dependen solo de `modo_agrupacion` +
universo de productos (`SLUG_DATOS`) -> se calculan una sola vez y se
comparten entre las 6 combinaciones de la grilla. Escalado/Optuna/train/submit
dependen de la combinacion completa (`slug_combo`).

In [ ]:
SLUG_DATOS = f"{PARAM['modo_agrupacion']}{'_todos' if not PARAM['solo_productos_target'] else '_target'}"


def slug_combo(metodo, tipo):
    return f"{SLUG_DATOS}__{metodo}__{tipo}"


PATH_PREPROCESADO = DIR_OUT / f"preprocesado__{SLUG_DATOS}.parquet"
PATH_FE            = DIR_OUT / f"fe__{SLUG_DATOS}.parquet"
PATH_FE_INFER      = DIR_OUT / f"fe_infer__{SLUG_DATOS}.parquet"

print(f"SLUG_DATOS   : {SLUG_DATOS}")
print(f"preprocesado : {PATH_PREPROCESADO.name}")
print(f"FE           : {PATH_FE.name} / {PATH_FE_INFER.name}")


## 3) Preprocesamiento

`agrupa_id` = producto o cliente x producto. Grid cliente x producto x periodo
con ceros completados entre nacimiento y muerte (via DuckDB: join + BETWEEN
sobre una tabla de periodos, vectorizado). Con `solo_productos_target=False`
(default nuevo) no se filtra a los ~780 productos a predecir: se usan todos.

In [ ]:
def _construir_preprocesado():
    print("Preprocesamiento: construyendo desde cero...")
    df_raw = pl.read_csv(
        DIR_RAW / "sell-in.txt.gz", separator="\t",
        schema_overrides={
            "periodo": pl.Int32, "customer_id": pl.Int32, "product_id": pl.Int32,
            "cust_request_qty": pl.Int32, "cust_request_tn": pl.Float32, "tn": pl.Float32,
        }
    )
    tb_apredecir = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t",
                               schema_overrides={"product_id": pl.Int32})
    tb_productos = pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")

    if PARAM["solo_productos_target"]:
        df_raw = df_raw.join(tb_apredecir, on="product_id", how="inner")
        print(f"Filtrado a {tb_apredecir.height} productos target: {df_raw.shape}")
    else:
        print(f"Usando TODOS los productos: {df_raw.shape}")

    MULT = 100_000
    if PARAM["modo_agrupacion"] == "producto":
        df_agr = (df_raw.group_by(["product_id", "periodo"])
                        .agg(pl.col("tn").sum().alias("tn"))
                        .with_columns(pl.col("product_id").cast(pl.Int64).alias("agrupa_id")))
    elif PARAM["modo_agrupacion"] == "cliente_producto":
        df_agr = (df_raw.group_by(["customer_id", "product_id", "periodo"])
                        .agg(pl.col("tn").sum().alias("tn"))
                        .with_columns((pl.col("customer_id").cast(pl.Int64) * MULT
                                      + pl.col("product_id").cast(pl.Int64)).alias("agrupa_id")))
    else:
        raise ValueError(f"modo_agrupacion invalido: {PARAM['modo_agrupacion']}")

    periodos_todos = sorted(df_agr["periodo"].unique().to_list())
    nace_muere = df_agr.group_by("agrupa_id").agg(
        pl.col("periodo").min().alias("nace"), pl.col("periodo").max().alias("muere")
    )

    con = duckdb.connect()
    con.register("nace_muere", nace_muere.to_pandas())
    con.register("periodos", pd.DataFrame({"periodo": periodos_todos}))
    grid = con.execute("""
        select nm.agrupa_id, p.periodo
        from nace_muere nm
        join periodos p on p.periodo between nm.nace and nm.muere
    """).pl()
    con.close()

    df_full = (grid.join(df_agr.select(["agrupa_id", "periodo", "tn"]), on=["agrupa_id", "periodo"], how="left")
                    .with_columns(pl.col("tn").fill_null(0.0))
                    .sort(["agrupa_id", "periodo"]))

    if PARAM["modo_agrupacion"] == "producto":
        df_full = df_full.with_columns(pl.col("agrupa_id").cast(pl.Int32).alias("product_id"))
    else:
        df_full = df_full.with_columns([
            (pl.col("agrupa_id") // MULT).cast(pl.Int32).alias("customer_id"),
            (pl.col("agrupa_id") % MULT).cast(pl.Int32).alias("product_id"),
        ])

    df_full = df_full.join(tb_productos, on="product_id", how="left")
    df_full = df_full.with_columns([
        pl.lit(PARAM["modo_agrupacion"]).alias("modo_agrupacion"),
        pl.lit(PARAM["solo_productos_target"]).alias("solo_productos_target"),
    ])

    print(f"Preprocesamiento listo: {df_full.shape}")
    return df_full


if PATH_PREPROCESADO.exists():
    print(f"[{SLUG_DATOS}] preprocesado ya existe -> RESUME (se saltea)")
    df_pre = pl.read_parquet(PATH_PREPROCESADO)
else:
    df_pre = _construir_preprocesado()
    escribir_atomico(lambda tmp: df_pre.write_parquet(tmp), PATH_PREPROCESADO)
    print(f"Guardado: {PATH_PREPROCESADO}")

print(f"df_pre: {df_pre.shape}")


## 4) Feature Engineering

Lags, medias moviles, vecinos (sustitutos/complementarios por correlacion),
hojas de Random Forest (tree embedding), share jerarquico + delta shares,
encoding categorico, y los dos targets (nivel y delta -- la eleccion de cual
usar es una palanca de mas adelante, no de aca).

In [ ]:
def _mes_corte_conservador(df):
    """Corte temporal conservador (N meses antes del ultimo periodo) que
    comparten vecinos y RF hojas, para no filtrar informacion futura."""
    corte_max = int(df["periodo"].max())
    anio, mes = divmod(corte_max, 100)
    total = anio * 12 + (mes - 1) - PARAM["meses_atras_corte_vecinos"]
    return (total // 12) * 100 + (total % 12) + 1


def _agregar_lags_y_medias(df):
    df = df.sort(["agrupa_id", "periodo"])
    lag_exprs = [pl.col("tn").shift(k).over("agrupa_id").alias(f"lag_{k}") for k in PARAM["lags"]]
    df = df.with_columns(lag_exprs)
    ma_exprs = [
        pl.col("tn").shift(1).rolling_mean(window_size=w, min_periods=1).over("agrupa_id").alias(f"media_movil_{w}")
        for w in PARAM["ventanas_media_movil"]
    ]
    df = df.with_columns(ma_exprs)
    return df


def _agregar_vecinos(df):
    mes_corte = _mes_corte_conservador(df)
    tot_prod = df.group_by(["product_id", "periodo"]).agg(pl.col("tn").sum().alias("tn_prod"))
    wide = (tot_prod.filter(pl.col("periodo") < mes_corte)
                    .pivot(on="product_id", index="periodo", values="tn_prod")
                    .sort("periodo").drop("periodo"))
    corr = wide.to_pandas().corr(method="spearman")

    n_vec = PARAM["n_vecinos"]
    filas = []
    for p in corr.columns:
        s = corr[p].drop(labels=[p]).dropna()
        if s.empty:
            continue
        for vecino, r in s.sort_values().head(n_vec).items():
            filas.append({"product_id": p, "tipo": "sustituto", "vecino_id": vecino})
        for vecino, r in s.sort_values(ascending=False).head(n_vec).items():
            filas.append({"product_id": p, "tipo": "complementario", "vecino_id": vecino})

    if not filas:
        print("aviso: no se pudieron calcular vecinos (muy pocos periodos antes del corte)")
        return df.with_columns([
            pl.lit(0.0).alias("tn_sustitutos_prom"),
            pl.lit(0.0).alias("tn_complementarios_prom"),
        ])

    vecinos = pl.from_pandas(pd.DataFrame(filas)).with_columns([
        pl.col("product_id").cast(pl.Int32), pl.col("vecino_id").cast(pl.Int32)
    ])
    feat = (vecinos.join(tot_prod.rename({"product_id": "vecino_id", "tn_prod": "tn_vecino"}),
                         on="vecino_id", how="left")
                   .group_by(["product_id", "tipo", "periodo"])
                   .agg(pl.col("tn_vecino").mean().alias("tn_vecino_prom")))
    feat_piv = feat.pivot(on="tipo", index=["product_id", "periodo"], values="tn_vecino_prom")
    for falt in ("sustituto", "complementario"):
        if falt not in feat_piv.columns:
            feat_piv = feat_piv.with_columns(pl.lit(None, dtype=pl.Float64).alias(falt))
    feat_piv = feat_piv.rename({"sustituto": "tn_sustitutos_prom", "complementario": "tn_complementarios_prom"})

    df = df.join(feat_piv, on=["product_id", "periodo"], how="left").with_columns([
        pl.col("tn_sustitutos_prom").fill_null(0.0),
        pl.col("tn_complementarios_prom").fill_null(0.0),
    ])
    print(f"Vecinos: corte={mes_corte}, {len(filas)} relaciones calculadas.")
    return df


def _agregar_share(df):
    cols_share = []
    for nivel in PARAM["niveles_share"]:
        if nivel not in df.columns:
            continue
        tot = df.group_by([nivel, "periodo"]).agg(pl.col("tn").sum().alias(f"tn_total_{nivel}"))
        df = df.join(tot, on=[nivel, "periodo"], how="left")
        df = df.with_columns(
            pl.when(pl.col(f"tn_total_{nivel}") > 0)
              .then(pl.col("tn") / pl.col(f"tn_total_{nivel}"))
              .otherwise(0.0)
              .alias(f"share_{nivel}")
        )
        cols_share.append(f"share_{nivel}")

    df = df.sort(["agrupa_id", "periodo"])
    for col in cols_share:
        for k in PARAM["lags_delta_share"]:
            df = df.with_columns(
                (pl.col(col) - pl.col(col).shift(k).over("agrupa_id")).alias(f"delta_{col}_{k}")
            )
    return df


def _agregar_targets(df):
    h = PARAM["horizonte"]
    df = df.sort(["agrupa_id", "periodo"])
    df = df.with_columns(pl.col("tn").shift(-h).over("agrupa_id").alias("tn_t2"))
    df = df.with_columns([
        pl.col("tn_t2").alias("target_nivel"),
        (pl.col("tn_t2") - pl.col("tn")).alias("target_delta"),
    ])
    return df


def _agregar_hojas_rf(df):
    mes_corte = _mes_corte_conservador(df)
    cols_id = {"agrupa_id", "product_id", "customer_id", "periodo"}
    cols_target = {"tn", "tn_t2", "target_nivel", "target_delta"}
    cols_meta = {"modo_agrupacion", "solo_productos_target"}
    prohibidas = cols_id | cols_target | cols_meta
    cat_cols = set(PARAM["cols_categoricas"])
    num_cols = [c for c in df.columns if c not in prohibidas and c not in cat_cols
               and df.schema[c] in (pl.Float32, pl.Float64, pl.Int32, pl.Int64)]

    tr = df.filter((pl.col("periodo") < mes_corte) & pl.col("target_nivel").is_not_null())
    if tr.height < 50:
        print("aviso: muy pocas filas para RF hojas, se saltea este feature.")
        return df

    X_tr = tr.select(num_cols).fill_null(0.0).to_numpy()
    y_tr = tr["target_nivel"].fill_null(0.0).to_numpy()

    rf = RandomForestRegressor(n_estimators=PARAM["n_arboles_rf"], max_depth=PARAM["profundidad_rf"],
                               min_samples_leaf=PARAM["min_hoja_rf"], n_jobs=-1, random_state=PARAM["semilla"])
    rf.fit(X_tr, y_tr)

    X_todo = df.select(num_cols).fill_null(0.0).to_numpy()
    hojas = rf.apply(X_todo)
    cols_hoja = [f"hoja_arbol_{i}" for i in range(hojas.shape[1])]
    df = df.with_columns([pl.Series(c, hojas[:, i].astype(str)) for i, c in enumerate(cols_hoja)])
    print(f"{len(cols_hoja)} columnas de hoja de RF agregadas (corte={mes_corte}, {len(num_cols)} features numericas).")
    return df


def _encodear_categoricas(df, cols_categoricas):
    for col in cols_categoricas:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col).cast(pl.Utf8).cast(pl.Categorical).to_physical().cast(pl.Int32).alias(col)
            )
    return df


def _construir_fe(df_pre):
    print("FE: construyendo desde cero...")
    df = df_pre
    df = _agregar_lags_y_medias(df)
    df = _agregar_vecinos(df)
    df = _agregar_share(df)
    df = _agregar_targets(df)
    df = _agregar_hojas_rf(df)
    # cols_categoricas efectivas: las declaradas + las hojas de RF que hayan salido (si salieron)
    cols_categoricas_fe = list(PARAM["cols_categoricas"]) + sorted(
        c for c in df.columns if c.startswith("hoja_arbol_")
    )
    df = _encodear_categoricas(df, cols_categoricas_fe)
    print(f"FE lista: {df.shape}")
    return df, cols_categoricas_fe


if PATH_FE.exists() and PATH_FE_INFER.exists():
    print(f"[{SLUG_DATOS}] FE ya existe -> RESUME (se saltea)")
    df_train_fe = pl.read_parquet(PATH_FE)
    df_infer_fe = pl.read_parquet(PATH_FE_INFER)
    COLS_CATEGORICAS_FE = list(PARAM["cols_categoricas"]) + sorted(
        c for c in df_train_fe.columns if c.startswith("hoja_arbol_")
    )
else:
    df_fe, COLS_CATEGORICAS_FE = _construir_fe(df_pre)
    df_train_fe = df_fe.filter(pl.col("tn_t2").is_not_null())
    df_infer_fe = df_fe.filter(pl.col("tn_t2").is_null())
    escribir_atomico(lambda tmp: df_train_fe.write_parquet(tmp), PATH_FE)
    escribir_atomico(lambda tmp: df_infer_fe.write_parquet(tmp), PATH_FE_INFER)
    print(f"Guardado: {PATH_FE.name} / {PATH_FE_INFER.name}")

print(f"train: {df_train_fe.shape}  infer: {df_infer_fe.shape}")
print(f"categoricas efectivas: {COLS_CATEGORICAS_FE}")


## 5) Estandarizar

Escala solo el **target** (`target_nivel`/`target_delta`), nunca las
features -- LightGBM es invariante a transformaciones monotonas de X, asi que
escalar features no cambiaria un solo corte del arbol; escalar el target si
cambia que peso tiene cada producto en la funcion de perdida.

- `mean`: divide por la media de TODO el historial pasado (ancla de largo plazo).
- `rolling_mean`: divide por la media de una ventana reciente corta (`ventana_escalado`)
  -- la idea es que la escala "acompañe" cambios de nivel recientes.
- `zscore`: `(target - media_ventana) / std_ventana`, misma ventana que `rolling_mean`.
  Para `tipo_target='delta'` no se recentra (un delta ya esta centrado en 0):
  solo se divide por el desvio.

Siempre con `shift(1)`: los parametros de escala de la fila en `periodo` usan
solo datos de periodos anteriores, nunca el propio.

In [ ]:
EPS = 1e-3


def escalar_target(y_raw, media_exp, media_vent, std_vent, metodo, es_delta):
    if metodo == "mean":
        denom = np.where(media_exp > EPS, media_exp, EPS)
        return y_raw / denom
    elif metodo == "rolling_mean":
        denom = np.where(media_vent > EPS, media_vent, EPS)
        return y_raw / denom
    elif metodo == "zscore":
        denom = np.where(std_vent > EPS, std_vent, EPS)
        return y_raw / denom if es_delta else (y_raw - media_vent) / denom
    raise ValueError(f"metodo_escalado invalido: {metodo}")


def desescalar_target(y_scaled, media_exp, media_vent, std_vent, metodo, es_delta):
    if metodo == "mean":
        denom = np.where(media_exp > EPS, media_exp, EPS)
        return y_scaled * denom
    elif metodo == "rolling_mean":
        denom = np.where(media_vent > EPS, media_vent, EPS)
        return y_scaled * denom
    elif metodo == "zscore":
        denom = np.where(std_vent > EPS, std_vent, EPS)
        return y_scaled * denom if es_delta else y_scaled * denom + media_vent
    raise ValueError(f"metodo_escalado invalido: {metodo}")


def _agregar_stats_escalado(df):
    df = df.sort(["agrupa_id", "periodo"])
    ventana_expandida = df.height  # mayor a cualquier serie por agrupa_id -> equivale a expanding
    w = PARAM["ventana_escalado"]
    return df.with_columns([
        pl.col("tn").shift(1).rolling_mean(window_size=ventana_expandida, min_periods=1)
          .over("agrupa_id").alias("media_expandida"),
        pl.col("tn").shift(1).rolling_mean(window_size=w, min_periods=1)
          .over("agrupa_id").alias("media_ventana"),
        pl.col("tn").shift(1).rolling_std(window_size=w, min_periods=2)
          .over("agrupa_id").alias("std_ventana"),
    ])


def _construir_escalado(metodo, tipo):
    target_col = "target_delta" if tipo == "delta" else "target_nivel"
    es_delta = (tipo == "delta")

    df_train_e = _agregar_stats_escalado(df_train_fe)
    df_infer_e = _agregar_stats_escalado(df_infer_fe)

    y_raw = df_train_e[target_col].fill_null(0.0).to_numpy()
    media_exp = df_train_e["media_expandida"].fill_null(0.0).to_numpy()
    media_vent = df_train_e["media_ventana"].fill_null(0.0).to_numpy()
    std_vent = df_train_e["std_ventana"].fill_null(0.0).to_numpy()
    y_scaled = escalar_target(y_raw, media_exp, media_vent, std_vent, metodo, es_delta)
    df_train_e = df_train_e.with_columns(pl.Series("target_scaled", y_scaled))

    return df_train_e, df_infer_e


def correr_estandarizar(metodo, tipo, slug):
    path = DIR_OUT / "escalado" / slug / "dataset_escalado.parquet"
    path_infer = DIR_OUT / "escalado" / slug / "dataset_escalado_infer.parquet"
    if path.exists() and path_infer.exists():
        print(f"  [{slug}] escalado ya existe -> RESUME (se saltea)")
        return pl.read_parquet(path), pl.read_parquet(path_infer)
    df_train_e, df_infer_e = _construir_escalado(metodo, tipo)
    escribir_atomico(lambda tmp: df_train_e.write_parquet(tmp), path)
    escribir_atomico(lambda tmp: df_infer_e.write_parquet(tmp), path_infer)
    print(f"  [{slug}] escalado guardado.")
    return df_train_e, df_infer_e


## 6) Optuna

Minimiza WAPE. El modelo entrena sobre el target **escalado**, pero en cada
trial se desescala la prediccion antes de medir WAPE -- el valor que Optuna
optimiza es siempre el WAPE real sobre toneladas, nunca sobre el espacio
escalado. Validacion por el esquema 'febreros' (validar contra los febreros
historicos, ya que el objetivo real es siempre febrero).

Storage de Optuna: SQLite local durante la corrida (el bucket via GCS-FUSE es
fragil para escritura concurrente), con backup atomico al bucket cada
`backup_cada_n_trials` trials. Si el estudio local no existe pero hay backup
en el bucket (ej. VM nueva), se restaura antes de arrancar -- si no, se pierde
el historial de trials aunque el backup exista.

In [ ]:
def calcular_wape(y_real, y_pred):
    y_real = np.asarray(y_real, dtype=np.float64)
    y_pred = np.maximum(np.asarray(y_pred, dtype=np.float64), 0.0)
    den = y_real.sum()
    return float("nan") if den == 0 else np.abs(y_real - y_pred).sum() / den


def calcular_pesos(periodos_serie, decay):
    if decay is None:
        return None
    periodos = sorted(periodos_serie.unique())
    idx = {p: i for i, p in enumerate(periodos)}
    n = len(periodos)
    return periodos_serie.map(lambda p: decay ** (n - 1 - idx[p])).values


def periodos_con_target_en_febrero(periodos, horizonte):
    pset = set(periodos)
    splits = []
    for p in periodos:
        anio, mes = divmod(p, 100)
        total = anio * 12 + (mes - 1) + horizonte
        objetivo = (total // 12) * 100 + (total % 12) + 1
        if objetivo % 100 == 2 and objetivo in pset:
            splits.append((p, p))
    return splits


def _cols_prohibidas():
    return {"agrupa_id", "product_id", "customer_id", "periodo", "tn", "tn_t2",
            "target_nivel", "target_delta", "target_scaled", "modo_agrupacion",
            "solo_productos_target", "media_expandida", "media_ventana", "std_ventana"}


def correr_optuna(df_train_e, metodo, tipo, slug):
    path_hiper = DIR_OUT / "optuna" / slug / "hiper.json"
    if path_hiper.exists():
        print(f"  [{slug}] hiper.json ya existe -> RESUME (se saltea)")
        return leer_json_reintentando(path_hiper)

    df_pd = df_train_e.to_pandas()
    features = [c for c in df_pd.columns if c not in _cols_prohibidas()]
    cat_features = [c for c in COLS_CATEGORICAS_FE if c in features]
    for c in cat_features:
        df_pd[c] = df_pd[c].astype("category")

    es_delta = (tipo == "delta")
    periodos = sorted(df_pd["periodo"].unique().tolist())
    if PARAM["esquema_val"] == "febreros":
        splits = periodos_con_target_en_febrero(periodos, PARAM["horizonte"])
        if not splits:
            splits = [(periodos[-2], periodos[-1])]
    else:
        splits = [(periodos[-2], periodos[-1])]

    db_local  = DIR_LOCAL / f"optuna__{slug}.db"
    db_bucket = DIR_OUT / "optuna" / slug / "study.db"
    if not db_local.exists() and db_bucket.exists():
        print(f"  [{slug}] restaurando study desde backup del bucket...")
        db_local.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(db_bucket, db_local)

    def objective(trial):
        params = {
            "objective": PARAM["objective_lgbm"], "metric": "mae", "verbosity": -1,
            "boosting_type": "gbdt", "seed": PARAM["semilla"],
        }
        if PARAM["regularizacion"] == "fuerte":
            params.update({
                "num_leaves": trial.suggest_int("num_leaves", 8, 64),
                "max_depth": trial.suggest_int("max_depth", 3, 7),
                "learning_rate": trial.suggest_float("learning_rate", 5e-3, 0.1, log=True),
                "n_estimators": trial.suggest_int("n_estimators", 100, 800),
                "min_child_samples": trial.suggest_int("min_child_samples", 30, 200),
            })
        else:
            params.update({
                "num_leaves": trial.suggest_int("num_leaves", 20, 300),
                "max_depth": trial.suggest_int("max_depth", 3, 12),
                "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
                "n_estimators": trial.suggest_int("n_estimators", 100, 2000),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            })

        errores = []
        for _corte, val_p in splits:
            tr = df_pd[df_pd["periodo"] < val_p]
            vl = df_pd[df_pd["periodo"] == val_p]
            if len(vl) == 0:
                continue
            w_tr = calcular_pesos(tr["periodo"], PARAM["decay_recencia"])

            modelo = lgb.LGBMRegressor(**params)
            modelo.fit(tr[features], tr["target_scaled"], sample_weight=w_tr,
                      eval_set=[(vl[features], vl["target_scaled"])],
                      categorical_feature=cat_features,
                      callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])

            pred_scaled = modelo.predict(vl[features])
            pred_real = desescalar_target(pred_scaled, vl["media_expandida"].values, vl["media_ventana"].values,
                                          vl["std_ventana"].values, metodo, es_delta)
            if es_delta:
                tn_actual = vl["tn"].values
                pred_nivel = tn_actual + pred_real
                real_nivel = tn_actual + vl["target_delta"].values
            else:
                pred_nivel = pred_real
                real_nivel = vl["target_nivel"].values

            errores.append(calcular_wape(real_nivel, pred_nivel))

        return float(np.mean(errores)) if errores else float("inf")

    def respaldar(study, trial):
        if trial.number % PARAM["backup_cada_n_trials"] == 0:
            escribir_atomico(lambda tmp: shutil.copy(db_local, tmp), db_bucket)

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=PARAM["semilla"]),
                                study_name=slug, storage=f"sqlite:///{db_local}", load_if_exists=True)
    print(f"  [{slug}] trials previos: {len(study.trials)} -- corriendo {PARAM['n_trials']} nuevos...")
    study.optimize(objective, n_trials=PARAM["n_trials"], callbacks=[respaldar])

    escribir_atomico(lambda tmp: shutil.copy(db_local, tmp), db_bucket)

    resultado = {
        "slug": slug, "modo_agrupacion": PARAM["modo_agrupacion"], "metodo_escalado": metodo,
        "tipo_target": tipo, "esquema_val": PARAM["esquema_val"], "metrica": "wape",
        "mejor_valor": study.best_value, "n_trials_total": len(study.trials),
        "features": features, "cat_features": cat_features, "hiperparametros": study.best_params,
    }
    escribir_atomico(lambda tmp: tmp.write_text(json.dumps(resultado, indent=2)), path_hiper)
    print(f"  [{slug}] Optuna listo: wape={study.best_value:.4f}")
    return resultado


## 7) Entrenamiento final + Submit

Entrena un modelo por semilla de `semillas_ensemble` con los hiperparametros
ganadores de Optuna, promedia predicciones, desescala, reconstruye nivel si el
target es delta, aplica `clip_min`. `resultado.json` es el marcador de combo
completo y la fuente del leaderboard. El submit queda guardado con su propio
marcador para no volver a subir el mismo combo si se reinicia despues de
submitear.

In [ ]:
def correr_entrenamiento_final(df_train_e, df_infer_e, hiper, metodo, tipo, slug):
    path_resultado = DIR_OUT / "final" / slug / "resultado.json"
    if path_resultado.exists():
        print(f"  [{slug}] resultado.json ya existe -> RESUME (se saltea)")
        return leer_json_reintentando(path_resultado)

    es_delta = (tipo == "delta")
    features = hiper["features"]
    cat_features = hiper["cat_features"]

    df_train_pd = df_train_e.to_pandas()
    df_infer_pd = df_infer_e.to_pandas()
    for c in cat_features:
        if c in df_train_pd.columns:
            df_train_pd[c] = df_train_pd[c].astype("category")
            df_infer_pd[c] = df_infer_pd[c].astype("category")

    w_train = calcular_pesos(df_train_pd["periodo"], PARAM["decay_recencia"])

    modelos = []
    for s in PARAM["semillas_ensemble"]:
        params_lgbm = {"objective": PARAM["objective_lgbm"], "metric": "mae", "verbosity": -1,
                       "boosting_type": "gbdt", "seed": s, **hiper["hiperparametros"]}
        m = lgb.LGBMRegressor(**params_lgbm)
        m.fit(df_train_pd[features], df_train_pd["target_scaled"], sample_weight=w_train,
             categorical_feature=cat_features)
        modelos.append(m)
        ruta_modelo = DIR_OUT / "final" / slug / f"modelo_seed{s}.txt"
        escribir_atomico(lambda tmp, m=m: m.booster_.save_model(str(tmp)), ruta_modelo)
    print(f"  [{slug}] {len(modelos)} modelo(s) entrenado(s).")

    preds = np.column_stack([m.predict(df_infer_pd[features]) for m in modelos])
    pred_scaled = preds.mean(axis=1)
    pred_real = desescalar_target(pred_scaled, df_infer_pd["media_expandida"].values,
                                  df_infer_pd["media_ventana"].values, df_infer_pd["std_ventana"].values,
                                  metodo, es_delta)
    pred_nivel = df_infer_pd["tn"].values + pred_real if es_delta else pred_real
    pred_nivel = np.maximum(pred_nivel, PARAM["clip_min"])

    resultado = {
        "slug": slug, "modo_agrupacion": PARAM["modo_agrupacion"], "metodo_escalado": metodo,
        "tipo_target": tipo, "metrica": "wape", "wape": hiper["mejor_valor"],
        "hiperparametros": hiper["hiperparametros"], "n_features": len(features),
        "semillas_ensemble": PARAM["semillas_ensemble"], "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    escribir_atomico(lambda tmp: tmp.write_text(json.dumps(resultado, indent=2)), path_resultado)

    tb_apredecir = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t",
                               schema_overrides={"product_id": pl.Int32}).to_pandas()
    df_pred = df_infer_pd[["product_id"]].copy()
    df_pred["tn"] = pred_nivel
    if PARAM["modo_agrupacion"] == "cliente_producto":
        df_pred = df_pred.groupby("product_id", as_index=False)["tn"].sum()
    # aseguramos que esten TODOS los productos pedidos, aunque a alguno le falte inferencia
    df_pred = tb_apredecir[["product_id"]].merge(df_pred, on="product_id", how="left")
    df_pred["tn"] = df_pred["tn"].fillna(PARAM["clip_min"])

    ruta_csv = DIR_OUT / "final" / slug / "submission.csv"
    escribir_atomico(lambda tmp: df_pred.to_csv(tmp, index=False), ruta_csv)

    print(f"  [{slug}] entrenamiento final listo. wape(val)={hiper['mejor_valor']:.4f}")
    return resultado


def correr_submit(slug):
    if not PARAM["submit"]:
        return
    path_log = DIR_OUT / "submissions" / slug / "submit_log.json"
    ruta_csv = DIR_OUT / "final" / slug / "submission.csv"
    if path_log.exists():
        print(f"  [{slug}] ya submiteado -> se saltea.")
        return
    if not ruta_csv.exists():
        print(f"  [{slug}] no hay submission.csv todavia, no se puede submitear.")
        return
    mensaje = f"pipe_final | {slug}"
    res = subprocess.run(["kaggle", "competitions", "submit", "-c", PARAM["kaggle_competition"],
                          "-f", str(ruta_csv), "-m", mensaje], capture_output=True, text=True)
    log = {"slug": slug, "returncode": res.returncode, "stdout": res.stdout, "stderr": res.stderr,
          "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
    escribir_atomico(lambda tmp: tmp.write_text(json.dumps(log, indent=2)), path_log)
    print(f"  [{slug}] submit: returncode={res.returncode}")


## 8) Grilla — recorrer combinaciones (resumible)

Un combo entero (escalado + Optuna + train + submit) se saltea si ya tiene
`resultado.json`. Los que faltan se corren; si el kernel se corta a mitad de
uno, al volver a correr esta celda los combos ya terminados se saltean
instantaneo y el que quedo a medias arranca de nuevo desde Optuna (que a su
vez resume sus propios trials via SQLite).

In [ ]:
def correr_combo(metodo, tipo):
    slug = slug_combo(metodo, tipo)
    path_resultado = DIR_OUT / "final" / slug / "resultado.json"
    if path_resultado.exists():
        print(f"[{slug}] ya tiene resultado.json -> RESUME (se saltea)")
        resultado = leer_json_reintentando(path_resultado)
    else:
        print(f"[{slug}] corriendo...")
        df_train_e, df_infer_e = correr_estandarizar(metodo, tipo, slug)
        hiper = correr_optuna(df_train_e, metodo, tipo, slug)
        resultado = correr_entrenamiento_final(df_train_e, df_infer_e, hiper, metodo, tipo, slug)
    correr_submit(slug)
    return resultado


resultados_grilla = []
for _metodo in PARAM["grilla_metodos_escalado"]:
    for _tipo in PARAM["grilla_tipos_target"]:
        resultados_grilla.append(correr_combo(_metodo, _tipo))

print(f"\nGrilla completa: {len(resultados_grilla)} combos procesados.")


## 9) Leaderboard

Escanea todos los `resultado.json` en el bucket y arma una tabla ordenada por
WAPE. Se puede correr en cualquier momento, incluso a mitad de la grilla, y
las veces que haga falta -- no modifica nada.

In [ ]:
filas = []
for p in sorted((DIR_OUT / "final").glob("*/resultado.json")):
    try:
        filas.append(leer_json_reintentando(p))
    except (OSError, json.JSONDecodeError):
        print(f"aviso: no se pudo leer {p}, se saltea.")

if not filas:
    print("(Todavia no hay combos terminados.)")
else:
    filas.sort(key=lambda f: (f["wape"] is None, f["wape"]))
    header = f"{'slug':45s} {'metodo':13s} {'target':7s} {'wape':8s} {'n_feat':6s} {'timestamp':20s}"
    print(header)
    print("-" * len(header))
    for f in filas:
        wape = f"{f['wape']:.4f}" if f["wape"] is not None else "?"
        print(f"{f['slug']:45s} {f['metodo_escalado']:13s} {f['tipo_target']:7s} {wape:8s} "
             f"{f['n_features']:6d} {f['timestamp']:20s}")
